In [ ]:
#Connect to drive
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

Create mixed patches using 2 interacting residues and 1 non interacting residue.

In [ ]:
base_fold_path = '/content/drive/MyDrive/abdb_v4'

import os
import numpy as np
import pandas as pd
import math
import itertools

# One-hot encoding for amino acid types
def encode_aa(aa):
    aa_types = ['ALA', 'CYS', 'ASP', 'GLU', 'PHE', 'GLY', 'HIS', 'ILE', 'LYS', 'LEU',
                'MET', 'ASN', 'PRO', 'GLN', 'ARG', 'SER', 'THR', 'VAL', 'TRP', 'TYR']
    return [1 if aa_type == aa else 0 for aa_type in aa_types]

# One-hot encoding for secondary structure
def encode_ss(ss):
    return [1 if ss == 'S' else 0, 1 if ss == 'L' else 0, 1 if ss == 'H' else 0]

# One-hot encoding for SaSaRate (Now 5 levels)
def encode_sasa(sasa):
    if sasa < 0.2:
        return [1, 0, 0, 0, 0]
    elif sasa < 0.4:
        return [0, 1, 0, 0, 0]
    elif sasa < 0.6:
        return [0, 0, 1, 0, 0]
    elif sasa < 0.8:
        return [0, 0, 0, 1, 0]
    else:
        return [0, 0, 0, 0, 1]

# Combine 3-d one-hot encoded vector for secondary structure with 5-d one-hot encoded vector for SaSaRate
# Resulting in a 15-d vector
def combine_ss_sasa(ss_encoded, sasa_encoded):
    combined = []
    for i in ss_encoded:
        for j in sasa_encoded:
            combined.append(i * j)
    return combined  # This should be a 15-d vector

# Calculate Euclidean distance between residues based on their side-chain coordinates
def calculate_distance(residue1, residue2):
    x1, y1, z1 = residue1['SCx'], residue1['SCy'], residue1['SCz']
    x2, y2, z2 = residue2['SCx'], residue2['SCy'], residue2['SCz']
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2 + (z2 - z1)**2)

# Process each fold's training set
for fold in range(1, 6):

    # Specify the directory and filename of the combined data CSV
    dir_name = os.path.join(base_fold_path, f'fold_{fold}', 'data')
    file_name = 'all_data_with_SI.csv'

    # Read the CSV file without setting the first column as an index
    combined_df = pd.read_csv(os.path.join(dir_name, file_name))
    filtered_combined_df = combined_df

    save_dir = '/content/drive/MyDrive/abdb_v4' + f'/fold_{fold}/data_processed_step1_with_SI/'
    os.makedirs(save_dir, exist_ok=True)

    # File paths for mixed patches
    txt_file_path = os.path.join(save_dir, 'cnn_mixed2p1n_3points.txt')

    # Create or overwrite the text file for mixed patches
    with open(txt_file_path, 'w') as f:
        f.write('')  # Writing an empty string to ensure the file is empty

    # Initialize an empty list to store all mixed patches
    mixed_patches = []

    # Initialize a counter for the number of mixed patches
    total_mixed_patches = 0

    # Loop through each group (protein)
    for name, group in filtered_combined_df.groupby('name'):
        # Filter residues with SaSaRate > 0.1
        filtered_group = group[group['SaSaRate'] > 0.1]

        # Separate interacting and non-interacting residues
        interacting_residues = filtered_group[filtered_group['Status'] == 'Interacting']
        non_interacting_residues = filtered_group[filtered_group['Status'] == 'Non-Interacting']

        patches = []

        # Loop over pairs of interacting residues
        for comb in itertools.combinations(interacting_residues.iterrows(), 2):
            index1, row1 = comb[0]
            index2, row2 = comb[1]

            # Check distance between the two interacting residues
            d12 = calculate_distance(row1, row2)

            if d12 < 8.5:  # If the distance is less than 8.5 Å

                # Loop over non-interacting residues
                for index3, row3 in non_interacting_residues.iterrows():
                    d13 = calculate_distance(row1, row3)
                    d23 = calculate_distance(row2, row3)

                    # Check distances between the non-interacting residue and both interacting residues
                    if d13 < 8.5 and d23 < 8.5:
                        patch_indices = [row1['ResID'], row2['ResID'], row3['ResID']]
                        patch_info = [
                            [row1['AA'], row1['SS'], row1['SaSaRate'], row1['SI']],
                            [row2['AA'], row2['SS'], row2['SaSaRate'], row2['SI']],
                            [row3['AA'], row3['SS'], row3['SaSaRate'], row3['SI']]
                        ]
                        patches.append({'indices': patch_indices, 'info': patch_info})

                        # Increment the counter
                        total_mixed_patches += 1

        # Output patches to the text file
        with open(txt_file_path, 'a') as f:
            f.write(f"Patches for protein {name}:\n")
            for patch in patches:
                f.write(f"indices: {patch['indices']}, info: {patch['info']}\n")

        # Convert each patch into a 15*20 matrix and append it to mixed_patches
        for patch in patches:
            patch_matrix = np.zeros((15, 20))  # Initialize a 15x20 matrix with zeros
            for idx, aa_info in enumerate(patch['info']):
                aa = aa_info[0]
                ss = aa_info[1]
                sasa = aa_info[2]
                si = aa_info[3]  # Extract the SI value

                aa_encoded = encode_aa(aa)
                ss_encoded = encode_ss(ss)
                sasa_encoded = encode_sasa(sasa)

                combined_info = combine_ss_sasa(ss_encoded, sasa_encoded)  # 15-d vector

                for i in range(20):
                    if aa_encoded[i] == 1:
                        patch_matrix[:, i] += np.array(combined_info) * si  # Multiply the SI value

            mixed_patches.append(patch_matrix)

    # Convert mixed_patches to a NumPy array
    mixed_patches_3points = np.array(mixed_patches)

    # Save the NumPy array for mixed patches
    np.save(os.path.join(save_dir, 'cnn_mixed2p1n_3points.npy'), mixed_patches_3points)

    # Print the number of mixed patches generated for this fold
    print(f"Total mixed patches generated for fold {fold}: {total_mixed_patches}")
    print(f"Saved mixed patches to cnn_mixed2p1n_3points.npy in {save_dir}")


Let's do a little sanity check to ensure that these mixed patches are properly generated.

In [ ]:
import pandas as pd
import os
import random

# Choose the fold you want to check
fold = 2  # Change this to the fold you're interested in

base_fold_path = '/content/drive/MyDrive/abdb_v4'

# Specify the directory and filename of the combined data CSV
dir_name = os.path.join(base_fold_path, f'fold_{fold}', 'data')
file_name = 'all_data_with_SI.csv'

# Read the CSV file
combined_df = pd.read_csv(os.path.join(dir_name, file_name))

# File path for mixed patches
save_dir = os.path.join(base_fold_path, f'fold_{fold}', 'data_processed_step1_with_SI')
txt_file_path = os.path.join(save_dir, 'cnn_mixed2p1n_3points.txt')

# Define 'nan' for eval()
nan = float('nan')

# Read the text file and collect all patches
patches = []
current_protein = None
with open(txt_file_path, 'r') as f:
    for line in f:
        line = line.strip()
        if line.startswith("Patches for protein"):
            current_protein = line.replace("Patches for protein ", "").replace(":", "")
        elif line.startswith("indices:"):
            # Parse the line to extract indices and info
            parts = line.split(", info: ")
            indices_part = parts[0].replace("indices: ", "")
            indices = eval(indices_part)  # Convert string representation of list to actual list
            info_part = parts[1]
            info = eval(info_part)  # Convert string representation of list to actual list
            patches.append({'protein': current_protein, 'indices': indices, 'info': info})

# Randomly select 3 patches
random_patches = random.sample(patches, 3)

# For each selected patch, get the ResIDs and retrieve the Status from combined_df
for idx, patch in enumerate(random_patches):
    print(f"Patch {idx+1}:")
    protein_name = patch['protein']
    res_ids = patch['indices']
    print(f"Protein: {protein_name}")
    print("Residues in the patch:")
    for res_id in res_ids:
        # Find the residue in combined_df
        residue = combined_df[(combined_df['name'] == protein_name) & (combined_df['ResID'] == res_id)]
        if not residue.empty:
            status = residue.iloc[0]['Status']
            aa = residue.iloc[0]['AA']
            ss = residue.iloc[0]['SS']
            sasa = residue.iloc[0]['SaSaRate']
            si = residue.iloc[0]['SI']
            print(f"ResID: {res_id}, AA: {aa}, SS: {ss}, SaSaRate: {sasa}, SI: {si}, Status: {status}")
        else:
            print(f"ResID: {res_id} not found in combined_df for protein {protein_name}")
    print("\n")

#Cool, it worked.

Generate  patches with 1 interacting and 2 non interacting residues


In [ ]:
base_fold_path = '/content/drive/MyDrive/abdb_v4'

import os
import numpy as np
import pandas as pd
import math
import itertools

# One-hot encoding for amino acid types
def encode_aa(aa):
    aa_types = ['ALA', 'CYS', 'ASP', 'GLU', 'PHE', 'GLY', 'HIS', 'ILE', 'LYS', 'LEU',
                'MET', 'ASN', 'PRO', 'GLN', 'ARG', 'SER', 'THR', 'VAL', 'TRP', 'TYR']
    return [1 if aa_type == aa else 0 for aa_type in aa_types]

# One-hot encoding for secondary structure
def encode_ss(ss):
    return [1 if ss == 'S' else 0, 1 if ss == 'L' else 0, 1 if ss == 'H' else 0]

# One-hot encoding for SaSaRate (Now 5 levels)
def encode_sasa(sasa):
    if sasa < 0.2:
        return [1, 0, 0, 0, 0]
    elif sasa < 0.4:
        return [0, 1, 0, 0, 0]
    elif sasa < 0.6:
        return [0, 0, 1, 0, 0]
    elif sasa < 0.8:
        return [0, 0, 0, 1, 0]
    else:
        return [0, 0, 0, 0, 1]

# Combine 3-d one-hot encoded vector for secondary structure with 5-d one-hot encoded vector for SaSaRate
# Resulting in a 15-d vector
def combine_ss_sasa(ss_encoded, sasa_encoded):
    combined = []
    for i in ss_encoded:
        for j in sasa_encoded:
            combined.append(i * j)
    return combined  # This should be a 15-d vector

# Calculate Euclidean distance between residues based on their side-chain coordinates
def calculate_distance(residue1, residue2):
    x1, y1, z1 = residue1['SCx'], residue1['SCy'], residue1['SCz']
    x2, y2, z2 = residue2['SCx'], residue2['SCy'], residue2['SCz']
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2 + (z2 - z1)**2)

# Process each fold's training set
for fold in range(1, 6):

    # Specify the directory and filename of the combined data CSV
    dir_name = os.path.join(base_fold_path, f'fold_{fold}', 'data')
    file_name = 'all_data_with_SI.csv'

    # Read the CSV file without setting the first column as an index
    combined_df = pd.read_csv(os.path.join(dir_name, file_name))
    filtered_combined_df = combined_df

    save_dir = os.path.join(base_fold_path, f'fold_{fold}/data_processed_step1_with_SI/')
    os.makedirs(save_dir, exist_ok=True)

    # File paths for mixed patches
    txt_file_path = os.path.join(save_dir, 'cnn_mixed1p2n_3points.txt')

    # Create or overwrite the text file for mixed patches
    with open(txt_file_path, 'w') as f:
        f.write('')  # Writing an empty string to ensure the file is empty

    # Initialize an empty list to store all mixed patches
    mixed_patches = []

    # Initialize a counter for the number of mixed patches
    total_mixed_patches = 0

    # Loop through each group (protein)
    for name, group in filtered_combined_df.groupby('name'):
        # Filter residues with SaSaRate > 0.1
        filtered_group = group[group['SaSaRate'] > 0.1]

        # Separate interacting and non-interacting residues
        interacting_residues = filtered_group[filtered_group['Status'] == 'Interacting']
        non_interacting_residues = filtered_group[filtered_group['Status'] == 'Non-Interacting']

        patches = []

        # Loop over pairs of non-interacting residues (MODIFIED HERE)
        for comb in itertools.combinations(non_interacting_residues.iterrows(), 2):
            index1, row1 = comb[0]
            index2, row2 = comb[1]

            # Check distance between the two non-interacting residues
            d12 = calculate_distance(row1, row2)

            if d12 < 8.5:  # If the distance is less than 8.5 Å

                # Loop over interacting residues
                for index3, row3 in interacting_residues.iterrows():
                    d13 = calculate_distance(row1, row3)
                    d23 = calculate_distance(row2, row3)

                    # Check distances between the interacting residue and both non-interacting residues
                    if d13 < 8.5 and d23 < 8.5:
                        patch_indices = [row1['ResID'], row2['ResID'], row3['ResID']]
                        patch_info = [
                            [row1['AA'], row1['SS'], row1['SaSaRate'], row1['SI']],
                            [row2['AA'], row2['SS'], row2['SaSaRate'], row2['SI']],
                            [row3['AA'], row3['SS'], row3['SaSaRate'], row3['SI']]
                        ]
                        patches.append({'indices': patch_indices, 'info': patch_info})

                        # Increment the counter
                        total_mixed_patches += 1

        # Output patches to the text file
        with open(txt_file_path, 'a') as f:
            f.write(f"Patches for protein {name}:\n")
            for patch in patches:
                f.write(f"indices: {patch['indices']}, info: {patch['info']}\n")

        # Convert each patch into a 15*20 matrix and append it to mixed_patches
        for patch in patches:
            patch_matrix = np.zeros((15, 20))  # Initialize a 15x20 matrix with zeros
            for idx, aa_info in enumerate(patch['info']):
                aa = aa_info[0]
                ss = aa_info[1]
                sasa = aa_info[2]
                si = aa_info[3]  # Extract the SI value

                aa_encoded = encode_aa(aa)
                ss_encoded = encode_ss(ss)
                sasa_encoded = encode_sasa(sasa)

                combined_info = combine_ss_sasa(ss_encoded, sasa_encoded)  # 15-d vector

                for i in range(20):
                    if aa_encoded[i] == 1:
                        patch_matrix[:, i] += np.array(combined_info) * si  # Multiply the SI value

            mixed_patches.append(patch_matrix)

    # Convert mixed_patches to a NumPy array
    mixed_patches_3points = np.array(mixed_patches)

    # Save the NumPy array for mixed patches
    np.save(os.path.join(save_dir, 'cnn_mixed1p2n_3points.npy'), mixed_patches_3points)

    # Print the number of mixed patches generated for this fold
    print(f"Total mixed patches generated for fold {fold}: {total_mixed_patches}")
    print(f"Saved mixed patches to cnn_mixed1p2n_3points.npy in {save_dir}")


Sanity check for the generated patches.

In [ ]:
import pandas as pd
import os
import random

# Choose the fold you want to check
fold = 2  # Change this to the fold you're interested in

base_fold_path = '/content/drive/MyDrive/abdb_v4'

# Specify the directory and filename of the combined data CSV
dir_name = os.path.join(base_fold_path, f'fold_{fold}', 'data')
file_name = 'all_data_with_SI.csv'

# Read the CSV file
combined_df = pd.read_csv(os.path.join(dir_name, file_name))

# File path for mixed patches
save_dir = os.path.join(base_fold_path, f'fold_{fold}', 'data_processed_step1_with_SI')
txt_file_path = os.path.join(save_dir, 'cnn_mixed1p2n_3points.txt')

# Define 'nan' for eval()
nan = float('nan')

# Read the text file and collect all patches
patches = []
current_protein = None
with open(txt_file_path, 'r') as f:
    for line in f:
        line = line.strip()
        if line.startswith("Patches for protein"):
            current_protein = line.replace("Patches for protein ", "").replace(":", "")
        elif line.startswith("indices:"):
            # Parse the line to extract indices and info
            parts = line.split(", info: ")
            indices_part = parts[0].replace("indices: ", "")
            indices = eval(indices_part)  # Convert string representation of list to actual list
            info_part = parts[1]
            info = eval(info_part)  # Convert string representation of list to actual list
            patches.append({'protein': current_protein, 'indices': indices, 'info': info})

# Randomly select 3 patches
random_patches = random.sample(patches, 3)

# For each selected patch, get the ResIDs and retrieve the Status from combined_df
for idx, patch in enumerate(random_patches):
    print(f"Patch {idx+1}:")
    protein_name = patch['protein']
    res_ids = patch['indices']
    print(f"Protein: {protein_name}")
    print("Residues in the patch:")
    for res_id in res_ids:
        # Find the residue in combined_df
        residue = combined_df[(combined_df['name'] == protein_name) & (combined_df['ResID'] == res_id)]
        if not residue.empty:
            status = residue.iloc[0]['Status']
            aa = residue.iloc[0]['AA']
            ss = residue.iloc[0]['SS']
            sasa = residue.iloc[0]['SaSaRate']
            si = residue.iloc[0]['SI']
            print(f"ResID: {res_id}, AA: {aa}, SS: {ss}, SaSaRate: {sasa}, SI: {si}, Status: {status}")
        else:
            print(f"ResID: {res_id} not found in combined_df for protein {protein_name}")
    print("\n")

#Cool, it worked.